In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

base_url = "https://books.toscrape.com/"
books = []

for page in range(1, 6):

    if page == 1:
        url = base_url
    else:
        url = base_url + "catalogue/page-" + str(page) + ".html"

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for item in soup.select("article.product_pod"):

        title = item.select_one("h3 a")["title"]
        price = item.select_one(".price_color").get_text(strip=True)

        rating = item.select_one(".star-rating")
        rating_text = rating.get("class")[1] if rating else "Unknown"

        availability = item.select_one(".availability").get_text(
            " ", strip=True
        )

        books.append({
            "title": title,
            "price": price,
            "star_rating": rating_text,
            "availability": availability
        })

df = pd.DataFrame(books)

print("Total books:", len(df))
print(df.head())

Total books: 100
                                   title    price star_rating availability
0                   A Light in the Attic  Â£51.77       Three     In stock
1                     Tipping the Velvet  Â£53.74         One     In stock
2                             Soumission  Â£50.10         One     In stock
3                          Sharp Objects  Â£47.82        Four     In stock
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock


In [12]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["price_gbp"] = (
    df["price"]
    .astype(str)
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.strip()
)

df["price_gbp"] = pd.to_numeric(
    df["price_gbp"],
    errors="coerce"
)

df["price_gbp"] = df["price_gbp"].fillna(
    df["price_gbp"].median()
)

df["rating"] = df["star_rating"].map(rating_map)

df["rating"] = df["rating"].fillna(
    df["rating"].median()
).astype(int)

df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

print(df[[
    "title",
    "price_gbp",
    "rating",
    "in_stock"
]].head())

print("\nData types:")
print(df[[
    "price_gbp",
    "rating",
    "in_stock"
]].dtypes)

print("\nMissing values:")
print(df[[
    "price_gbp",
    "rating",
    "in_stock"
]].isnull().sum())

                                   title  price_gbp  rating  in_stock
0                   A Light in the Attic      51.77       3      True
1                     Tipping the Velvet      53.74       1      True
2                             Soumission      50.10       1      True
3                          Sharp Objects      47.82       4      True
4  Sapiens: A Brief History of Humankind      54.23       5      True

Data types:
price_gbp    float64
rating         int64
in_stock        bool
dtype: object

Missing values:
price_gbp    0
rating       0
in_stock     0
dtype: int64


In [13]:
rate = 105.50

df["price_inr"] = (
    df["price_gbp"] * rate
).round(2)

print("Fixed conversion rate: 1 GBP = 105.50 INR")
print(df[[
    "title",
    "price_gbp",
    "price_inr"
]].head())

print("\nTotal books:", len(df))

Fixed conversion rate: 1 GBP = 105.50 INR
                                   title  price_gbp  price_inr
0                   A Light in the Attic      51.77    5461.74
1                     Tipping the Velvet      53.74    5669.57
2                             Soumission      50.10    5285.55
3                          Sharp Objects      47.82    5045.01
4  Sapiens: A Brief History of Humankind      54.23    5721.26

Total books: 100


In [17]:
import sqlite3

conn = sqlite3.connect("books.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

cursor.execute("""
INSERT INTO categories (category_name)
VALUES ('All Products')
""")

cursor.execute("""
SELECT category_id
FROM categories
WHERE category_name = 'All Products'
""")

category_id = cursor.fetchone()[0]

for _, row in df.iterrows():

    cursor.execute("""
    INSERT INTO books
    (
        title,
        price_gbp,
        price_inr,
        rating,
        in_stock,
        category_id
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["title"],
        float(row["price_gbp"]),
        float(row["price_inr"]),
        int(row["rating"]),
        int(row["in_stock"]),
        category_id
    ))

conn.commit()

print("Database created successfully")
print("Books inserted:", len(df))
print("Category:", "All Products")

Database created successfully
Books inserted: 100
Category: All Products


In [18]:
# Query 1 - SELECT and WHERE
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4
"""
result1 = pd.read_sql(query1, conn)

# Query 2 - ORDER BY and LIMIT
query2 = """
SELECT title, price_gbp, price_inr
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""
result2 = pd.read_sql(query2, conn)

# Query 3 - DISTINCT
query3 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating
"""
result3 = pd.read_sql(query3, conn)

# Query 4 - BETWEEN
query4 = """
SELECT title, price_gbp, rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
"""
result4 = pd.read_sql(query4, conn)

# Query 5 - JOIN and IN
query5 = """
SELECT
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
WHERE c.category_name IN (
    SELECT category_name
    FROM categories
    LIMIT 3
)
ORDER BY b.rating DESC
LIMIT 10
"""
result5 = pd.read_sql(query5, conn)

print("QUERY 1")
print(result1)

print("\nQUERY 2")
print(result2)

print("\nQUERY 3")
print(result3)

print("\nQUERY 4")
print(result4)

print("\nQUERY 5 - JOIN")
print(result5)

# Read the tables into pandas
books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)

# Reproduce the JOIN using pandas
merged = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

merged = merged[[
    "title",
    "price_gbp",
    "price_inr",
    "rating",
    "category_name"
]]

merged = merged.sort_values(
    by="rating",
    ascending=False
).head(10)

print("\nPANDAS MERGE RESULT")
print(merged)

conn.close()

QUERY 1
                                                title  price_gbp  rating
0                                       Sharp Objects      47.82       4
1               Sapiens: A Brief History of Humankind      54.23       5
2   The Dirty Little Secrets of Getting Your Dream...      33.34       4
3   The Boys in the Boat: Nine Americans and Their...      22.60       4
4                               Shakespeare's Sonnets      20.66       4
5                                         Set Me Free      17.46       5
6   Scott Pilgrim's Precious Little Life (Scott Pi...      52.29       5
7                           Rip it Up and Start Again      35.02       5
8                          Chase Me (Paris Nights #2)      25.27       5
9                                          Black Dust      34.53       5
10  Worlds Elsewhere: Journeys Around Shakespeareâ...      40.30       5
11                                     Wall and Piece      44.18       4
12  The Four Agreements: A Practical Guide 